# Treino D20 V1

Notebook de treino supervisionado (1..20) usando OpenCV + extração de features de ROI do número.

In [ ]:
from pathlib import Path
import sys
from datetime import datetime, timezone

import numpy as np
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.config import ProjectConfig
from src.data_io import load_images_and_labels
from src.cv_pipeline import image_to_feature
from src.modeling import train_centroid_classifier, evaluate_classifier, save_model_bundle

cfg = ProjectConfig()
print(f'Projeto: {PROJECT_ROOT}')
print(f'Input: {cfg.input_dir}')
print(f'Modelo: {cfg.model_path}')

In [ ]:
images, labels, paths = load_images_and_labels(PROJECT_ROOT / cfg.input_dir)
print(f'Amostras carregadas: {len(images)}')
if not images:
    raise RuntimeError('Nenhuma imagem encontrada em input/d20.')

print('Exemplos de arquivos:')
for p, label in list(zip(paths, labels))[:5]:
    print(f'- {p.name} -> label={label}')

In [ ]:
features = []
labels_array = []
sample_rois = []

for image, label in zip(images, labels):
    feature, meta, number_roi = image_to_feature(image)
    features.append(feature)
    labels_array.append(label)
    if len(sample_rois) < 6:
        sample_rois.append((number_roi, label, meta))

X = np.stack(features).astype(np.float32)
y = np.array(labels_array, dtype=np.int32)

print('Shape features:', X.shape)
print('Classes presentes:', sorted(set(y.tolist())))

In [ ]:
rng = np.random.default_rng(cfg.random_seed)
idx = rng.permutation(len(X))
split = int(len(X) * cfg.train_ratio)

if split <= 0 or split >= len(X):
    raise RuntimeError('Ajuste train_ratio para manter treino e validacao com amostras.')

train_idx = idx[:split]
test_idx = idx[split:]

X_train, y_train = X[train_idx], y[train_idx]
X_test, y_test = X[test_idx], y[test_idx]

classifier = train_centroid_classifier(X_train, y_train)
train_metrics = evaluate_classifier(classifier, X_train, y_train)
test_metrics = evaluate_classifier(classifier, X_test, y_test)

print('Acuracia treino:', round(train_metrics['accuracy'], 4))
print('Acuracia validacao:', round(test_metrics['accuracy'], 4))
print('Confianca media validacao:', round(test_metrics['mean_confidence'], 4))

In [ ]:
cm = np.array(test_metrics['confusion_matrix'])
classes = test_metrics['classes']

plt.figure(figsize=(7, 6))
plt.imshow(cm, cmap='Blues')
plt.title('Matriz de confusao - validacao')
plt.xlabel('Predito')
plt.ylabel('Real')
plt.xticks(range(len(classes)), classes, rotation=90)
plt.yticks(range(len(classes)), classes)
plt.colorbar()
plt.tight_layout()
plt.show()

In [ ]:
bundle_metrics = {
    'train': train_metrics,
    'test': test_metrics,
}

metadata = {
    'model_version': cfg.model_version,
    'trained_at': datetime.now(timezone.utc).isoformat(),
    'dataset_size': int(len(X)),
}

save_model_bundle(
    PROJECT_ROOT / cfg.model_path,
    classifier=classifier,
    metrics=bundle_metrics,
    preprocessing={'feature_size': [32, 32]},
    metadata=metadata,
)

print(f'Modelo salvo em: {PROJECT_ROOT / cfg.model_path}')